# GradScope — Autograd & Manifold Experiments

Welcome to the **GradScope** scratchpad. This notebook documents the mathematical derivation and implementation verification of our from-scratch scalar autograd engine.

## 1. Scalar Autograd: The Chain Rule

Reverse-mode automatic differentiation computes partial derivatives $\frac{\partial L}{\partial v_i}$ for all nodes $v_i$ by traversing the computational DAG in reverse topological order.

By the multivariable chain rule:
$$\frac{\partial L}{\partial x} = \sum_{y \in \text{children}(x)} \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$$

Let's test this in code using our pure Python engine.

In [ ]:
import sys
sys.path.append('..')
from engine.value import Value

# Construct a computational DAG
a = Value(2.0, _label='a')
b = Value(-3.0, _label='b')
c = Value(10.0, _label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, _label='f')
L = d * f; L.label = 'L'

# Run reverse-mode autodiff
L.backward()

print(f"L.data = {L.data}")
print(f"dL/da = {a.grad} (expected: b * f = -3 * -2 = 6.0)")
print(f"dL/db = {b.grad} (expected: a * f = 2 * -2 = -4.0)")
print(f"dL/dc = {c.grad} (expected: 1 * f = -2.0)")

## 2. Numerical Gradient Check

We verify exact equivalence with central finite differences:
$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}, \quad h = 10^{-6}$$

In [ ]:
h = 1e-6

def eval_L(a_val, b_val, c_val, f_val):
    return ((a_val * b_val) + c_val) * f_val

num_dL_da = (eval_L(2.0 + h, -3.0, 10.0, -2.0) - eval_L(2.0 - h, -3.0, 10.0, -2.0)) / (2 * h)
print(f"Analytical dL/da: {a.grad:.6f}, Numerical dL/da: {num_dL_da:.6f}")
assert abs(a.grad - num_dL_da) < 1e-5, "Gradient check failed!"
print("Autograd gradient matches numerical derivative precisely.")

## 3. Training an MLP on 2D XOR

A single linear layer provably cannot separate XOR. We train an MLP with hidden layers [4, 4] to observe how non-linear activations untangle the geometry.

In [ ]:
from engine.nn import MLP
from datasets.toy_datasets import make_xor

model = MLP(2, [4, 4, 1], nonlin='tanh', output_nonlin='tanh')
X, Y = make_xor(n_points=60, noise=0.05, seed=42)
y_targets = [-1.0 if y == 0 else 1.0 for y in Y]

print(f"Training MLP with {len(model.parameters())} parameters on {len(X)} samples...")

for epoch in range(50):
    model.zero_grad()
    total_loss = Value(0.0)
    correct = 0
    
    for x_pt, y_tgt in zip(X, y_targets):
        pred, coords, _ = model.forward_with_intermediates(x_pt)
        loss_i = (pred - y_tgt) ** 2
        total_loss = total_loss + loss_i
        if (pred.data > 0) == (y_tgt > 0):
            correct += 1
            
    loss = total_loss * (1.0 / len(X))
    loss.backward()
    
    # SGD update
    for p in model.parameters():
        p.data -= 0.15 * p.grad
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {loss.data:.4f} | Accuracy: {(correct / len(X)) * 100:.1f}%")